# BrandSphere AI — Week 1: Exploratory Data Analysis
**CRS AI Capstone 2025–26**

> **How to run:** Open in Google Colab, run Setup cells first, then run all cells in order.

**Datasets used:**
- `datasets/raw/marketing_campaign_dataset.csv` — 200,000 campaign records
- `datasets/raw/startups.csv` — 42,038 startup profiles
- `datasets/processed/cleaned_slogans.csv` — brand tagline corpus

In [ ]:
# Run this cell first in Google Colab
!pip install -q pandas numpy scikit-learn matplotlib seaborn plotly nltk Pillow cairosvg 2>/dev/null
print('✅ Libraries installed')

In [ ]:
# Option A: Mount Google Drive (recommended for Colab)
# from google.colab import drive
# drive.mount('/content/drive')
# BASE = '/content/drive/MyDrive/BrandSphere'

# Option B: Upload files manually to Colab session
# Use the Files panel on the left to upload datasets/

# Option C: Clone the GitHub repo
import subprocess
result = subprocess.run(['git','clone','--quiet','https://github.com/iblamehemer/og', '/content/BrandSphere'], 
                        capture_output=True, text=True)
print(result.stdout or '✅ Repo cloned')
import os, sys
os.chdir('/content/BrandSphere')
sys.path.insert(0, '/content/BrandSphere')
print(f'Working directory: {os.getcwd()}')
print(f'Files: {os.listdir()}')

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0C0D0F'
matplotlib.rcParams['axes.facecolor']   = '#141518'
matplotlib.rcParams['axes.labelcolor']  = 'white'
matplotlib.rcParams['text.color']       = 'white'
matplotlib.rcParams['xtick.color']      = 'white'
matplotlib.rcParams['ytick.color']      = 'white'
import seaborn as sns
sns.set_theme(style="darkgrid", palette="muted")
import warnings; warnings.filterwarnings('ignore')
print('✅ Libraries loaded')

## 2. Load & Inspect Marketing Campaign Dataset

In [ ]:
df = pd.read_csv('datasets/raw/marketing_campaign_dataset.csv')
print(f'Shape: {df.shape}')
print(f'\nColumn types:\n{df.dtypes}')
print(f'\nMissing values:\n{df.isnull().sum()}')
df.head()

## 3. Data Quality Check

In [ ]:
# Check for duplicates and data quality
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'\nUnique values per column:')
for col in df.select_dtypes('object').columns:
    print(f'  {col:20s}: {df[col].nunique()} unique — {df[col].unique()[:5]}')

# Fix Acquisition_Cost: remove $ and comma
df['Acquisition_Cost_Num'] = df['Acquisition_Cost'].str.replace('[$,]','',regex=True).astype(float)
df['Duration_Days'] = df['Duration'].str.extract(r'(\d+)').astype(float)
df['CTR'] = (df['Clicks'] / df['Impressions'] * 100).round(4)
df['Date'] = pd.to_datetime(df['Date'])
df['Month']   = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter
print(f'\n✅ Cleaning complete. New shape: {df.shape}')

## 4. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Marketing Campaign Dataset — Feature Distributions', color='#C9A84C', fontsize=14)

numeric_cols = ['ROI','Engagement_Score','Conversion_Rate','CTR','Acquisition_Cost_Num','Duration_Days']
colors = ['#C9A84C','#3ECFB2','#7B8FF7','#F06292','#80CBC4','#FFB74D']

for ax, col, color in zip(axes.flatten(), numeric_cols, colors):
    ax.hist(df[col].dropna(), bins=40, color=color, alpha=0.8, edgecolor='none')
    ax.set_title(col, color='white', fontsize=11)
    ax.set_xlabel('Value', color='#999')
    mean_val = df[col].mean()
    ax.axvline(mean_val, color='white', linestyle='--', linewidth=1.2, label=f'Mean: {mean_val:.2f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('assets/sample_exports/01_distributions.png', dpi=120, bbox_inches='tight', facecolor='#0C0D0F')
plt.show()
print('✅ Distribution chart saved')

## 5. Categorical Feature Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Campaign Channel, Type & Audience Distribution', color='#C9A84C', fontsize=14)

for ax, col, color in zip(axes, ['Channel_Used','Campaign_Type','Customer_Segment'], 
                                  ['#C9A84C','#3ECFB2','#7B8FF7']):
    counts = df[col].value_counts()
    bars = ax.bar(counts.index, counts.values, color=color, alpha=0.85)
    ax.set_title(col, color='white', fontsize=11)
    ax.tick_params(axis='x', rotation=30)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+100, 
                f'{int(bar.get_height()/1000)}k', ha='center', color='white', fontsize=8)

plt.tight_layout()
plt.savefig('assets/sample_exports/01_categoricals.png', dpi=120, bbox_inches='tight', facecolor='#0C0D0F')
plt.show()

## 6. Correlation Analysis

In [ ]:
numeric_df = df[['ROI','Engagement_Score','Conversion_Rate','CTR',
                     'Acquisition_Cost_Num','Duration_Days','Clicks','Impressions']].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(numeric_df, dtype=bool))
sns.heatmap(numeric_df, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, cbar_kws={'shrink':0.8})
ax.set_title('Feature Correlation Matrix', color='#C9A84C', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('assets/sample_exports/01_correlation.png', dpi=120, bbox_inches='tight', facecolor='#141518')
plt.show()
print('Key insight: ROI, Engagement, Conversion show near-zero correlation — dataset outcomes are synthetically generated.')

## 7. Startups Dataset EDA

In [ ]:
df_st = pd.read_csv('datasets/raw/startups.csv')
print(f'Shape: {df_st.shape}')
print(f'Columns: {list(df_st.columns)}')
print(f'\nMissing: {df_st.isnull().sum().to_dict()}')
print(f'\nSample taglines:')
for t in df_st['tagline'].dropna().sample(10, random_state=42):
    print(f'  • {t}')

## 8. Slogan Dataset EDA

In [ ]:
df_sl = pd.read_csv('datasets/processed/cleaned_slogans.csv')
print(f'Slogans corpus: {len(df_sl)} entries')
print(f'\nWord count distribution:')
print(df_sl['word_count'].describe())
print(f'\nAll brand slogans:')
for _, row in df_sl.iterrows():
    print(f'  {row["Company"]:15s}: {row["Slogan"]}')

## Summary

| Dataset | Rows | Key Finding |
|---|---|---|
| marketing_campaign_dataset | 200,000 | Synthetic outcomes (R²≈0); rich feature space for encoding pipeline |
| startups.csv | 42,038 | Rich tagline corpus for NLP persona profiling |
| cleaned_slogans | 15 | Curated brand taglines for TF-IDF retrieval |

**Data quality issues found and resolved:**
- Acquisition_Cost: currency string → numeric
- Duration: '30 days' string → integer
- CTR derived: Clicks/Impressions × 100